# 5 — Fail-closed validation, analysis, and export
Analysis refuses partial or contaminated results. Primary estimates use only fresh seeds 46–109. Download the archive immediately after creation.


In [ ]:
import csv, hashlib, json, os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3_new"; PY=Path.home()/"venv-stage1-ood/bin/python"; A=OUT/"analysis"
env=os.environ.copy(); env["MPLBACKEND"]="Agg"
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage3_new","--manifest",str(OUT/"stage3_new_manifest.csv"),"--output-dir",str(OUT)],cwd=R,env=env,check=True)
rows=list(csv.DictReader(open(OUT/"stage3_new_episode_results.csv")))
assert len(rows)==1944 and len({r['run_id'] for r in rows})==1944; assert {int(r['seed']) for r in rows}==set(range(46,82)); assert not ({*range(14,22)} & {int(r['seed']) for r in rows})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.analyze_stage3_new","--results",str(OUT/"stage3_new_episode_results.csv"),"--manifest",str(OUT/"stage3_new_manifest.csv"),"--output-dir",str(A),"--bootstrap-replicates","10000","--bootstrap-seed","20260826"],cwd=R,env=env,check=True)
required={"stage3_new_four_cell_by_candidate_horizon.csv","stage3_new_interaction_by_candidate_horizon.csv","stage3_new_object_layout_cross_task_contrasts.csv","stage3_new_bootstrap_intervals.csv","STAGE_3_NEW_OBSERVATIONS.md"}
missing=[name for name in required if not ((A/name).exists() or (OUT/name).exists())]; assert not missing, missing
print(*sorted(p.name for p in A.iterdir()),sep="\n")


In [ ]:
# Persist final provenance only after validation and analysis pass.
manifest=OUT/"stage3_new_manifest.csv"; results=OUT/"stage3_new_episode_results.csv"
sha=lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
provenance={"stage":"stage_3_new_high_power_replication","complete":True,"physical_episode_count":1944,"seed_block":[46,81],"bootstrap_replicates":10000,"bootstrap_rng_seed":20260826,"manifest_sha256":sha(manifest),"episode_results_sha256":sha(results),"old_stage3_stage3b_rows_pooled":False,"shared_id_rows_duplicated":False}
(OUT/"stage3_new_provenance.json").write_text(json.dumps(provenance,indent=2,sort_keys=True)+"\n")
subprocess.run([str(Path.home()/"venv-stage1-id/bin/pip"),"freeze"],stdout=open(OUT/"pip_freeze_id.txt","w"),check=True)
subprocess.run([str(Path.home()/"venv-stage1-ood/bin/pip"),"freeze"],stdout=open(OUT/"pip_freeze_ood.txt","w"),check=True)
archive=Path.home()/"stage3_new_results.tar.gz"; subprocess.run(["tar","-czf",str(archive),"-C",str(Path.home()),"stage3_new"],check=True)
print(archive,archive.stat().st_size,"bytes"); print("sha256",sha(archive)); print("DOWNLOAD THIS ARCHIVE NOW")
